# 🧱 Example 1: Basic CTE Structure - Simple Ordering

🔍 **Why use a CTE here?**  
It demonstrates how you can use the CTE definition (and associated query) as a "temp table". The main query can they be usedto select from the CTE by name.

In [5]:
USE Northwind2025;
GO

-- Total Sales by Employees
WITH EmployeeSales AS (
    SELECT e.EmployeeID
          ,e.FirstName + ' ' + e.LastName AS EmployeeName
          ,SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) AS TotalSales
    FROM emp.Employees e
    INNER JOIN sales.Orders o 
     ON e.EmployeeID = o.EmployeeID
    INNER JOIN sales.OrderDetails od ON o.OrderID = od.OrderID
    GROUP BY e.EmployeeID, e.FirstName, e.LastName
)
SELECT * 
FROM EmployeeSales
ORDER BY TotalSales DESC
;

Commands completed successfully.

(9 rows affected)

Total execution time: 00:00:00.124

EmployeeID,EmployeeName,TotalSales
4,Margaret Peacock,232890.84594726562
3,Janet Leverling,202812.84279346466
1,Nancy Davolio,192107.60432052612
2,Andrew Fuller,166537.75497817993
8,Laura Callahan,126862.27770423889
7,Robert King,124568.23538208008
9,Anne Dodsworth,77308.06670951843
6,Michael Suyama,73913.12924385071
5,Steven Buchanan,68792.28253936768


In [4]:
-- You can use use an optional field output after the CTE name is declared
-- This alleviates using column aliases for each row

USE Northwind2025;
GO

-- Total Sales by Employees
WITH EmployeeSales (Id, Employee, Sales) AS (
    SELECT e.EmployeeID
          ,e.FirstName + ' ' + e.LastName 
          ,SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) 
    FROM emp.Employees e
    INNER JOIN sales.Orders o 
     ON e.EmployeeID = o.EmployeeID
    INNER JOIN sales.OrderDetails od ON o.OrderID = od.OrderID
    GROUP BY e.EmployeeID, e.FirstName, e.LastName
)
SELECT * 
FROM EmployeeSales
ORDER BY Sales DESC
;

Commands completed successfully.

(9 rows affected)

Total execution time: 00:00:00.033

Id,Employee,Sales
4,Margaret Peacock,232890.84594726562
3,Janet Leverling,202812.84279346466
1,Nancy Davolio,192107.60432052612
2,Andrew Fuller,166537.75497817993
8,Laura Callahan,126862.27770423889
7,Robert King,124568.23538208008
9,Anne Dodsworth,77308.06670951843
6,Michael Suyama,73913.12924385071
5,Steven Buchanan,68792.28253936768


# 🧠 Example 2: Breaking Down Aggregations — Average Sales per Salesperson

🔍 **Why use a CTE here?**  
It separates the aggregation logic (`AVG(Total)`) from the final join, making the query easier to read and maintain.

In [ ]:
USE Northwind2025;
GO

WITH EmployeeSales AS (
    SELECT sal.EmployeeID
          ,AVG(sal.TotalSale) AS AvgSale
    FROM fact.Sales AS sal
    GROUP BY sal.EmployeeID
)
SELECT emp.EmployeeName
      ,es.AvgSale
FROM EmployeeSales es
INNER JOIN dim.Employees emp
 ON es.EmployeeID = emp.EmployeeID
ORDER BY es.AvgSale DESC
;

# 📊 Example 3: Joining Filtering Before Aggregating — Canada, UK, USA Customer Ranking based on Sales

🔍 **Why use a CTE here?**  
This can all be done in one query, but this seperates out the two complexities within the query:  

- multiple joins
- aggregation

In [6]:
WITH ActiveCustomers (Id, Customer, [Product], ProductCategory, Sales) 
  AS(
    SELECT cus.CustomerID
          ,cus.CompanyName
          ,prod.ProductName
          ,cat.CategoryName
          ,od.Quantity * od.UnitPrice * (1-od.Discount)
    FROM sales.Customers cus
        INNER JOIN sales.Orders ord
         ON cus.CustomerID = ord.CustomerID
        INNER JOIN sales.OrderDetails od
         ON ord.OrderID = od.OrderID
        INNER JOIN prod.Products prod
         on od.ProductID = prod.ProductID
        INNER JOIN prod.Categories cat
         on prod.CategoryID = cat.CategoryID

    WHERE cus.Country in ('UK', 'USA', 'Canada')
)
SELECT TOP 10 with ties ac.Customer
      ,round(sum(ac.Sales),2) as Sales
FROM ActiveCustomers ac
GROUP BY ac.Customer
ORDER BY Sales desc
;

(10 rows affected)

Total execution time: 00:00:00.011

Customer,Sales
Save-a-lot Markets,104361.95
Rattlesnake Canyon Grocery,51097.8
Mère Paillarde,28872.19
White Clover Markets,27363.61
Bottom-Dollar Markets,20801.6
Great Lakes Food Market,18507.45
Seven Seas Imports,16215.33
Old World Delicatessen,15177.46
Eastern Connection,14761.04
Around the Horn,13390.65


In [10]:
-- This allows the report author to alter the main query, reusing the CTE definition...
WITH ActiveCustomers (Id, Customer, [Product], ProductCategory, Sales) 
  AS(
    SELECT cus.CustomerID
          ,cus.CompanyName
          ,prod.ProductName
          ,cat.CategoryName
          ,od.Quantity * od.UnitPrice * (1-od.Discount)
    FROM sales.Customers cus
        INNER JOIN sales.Orders ord
         ON cus.CustomerID = ord.CustomerID
        INNER JOIN sales.OrderDetails od
         ON ord.OrderID = od.OrderID
        INNER JOIN prod.Products prod
         on od.ProductID = prod.ProductID
        INNER JOIN prod.Categories cat
         on prod.CategoryID = cat.CategoryID

    WHERE cus.Country in ('UK', 'USA', 'Canada')
)
SELECT ac.ProductCategory
      ,sum(ac.Sales) as 'TotalSales'
FROM ActiveCustomers ac
GROUP BY ac.ProductCategory
HAVING sum(ac.Sales) >= 30000
ORDER BY TotalSales desc
;

(6 rows affected)

Total execution time: 00:00:00.012

ProductCategory,TotalSales
Beverages,79073.13540649414
Dairy Products,61617.67004776001
Confections,53990.45293426514
Meat/Poultry,51744.26963043213
Seafood,32623.212468147278
Grains/Cereals,30021.014980316162


# 📈 Example 4: Multi-step Analysis — Orders with Freight Charges \>= Avg Freight Charge for 2024

🔍 **Why use a CTE here?**  
This can all be done in one query with subquery to enumerate the avg freight charges for 2024. What this does is break down each analytical component into a "daisy chain" of queries. 

<span style="color: var(--vscode-foreground);">Then the main query can use each of these queries as needed using simple joins, orders, etc. The report auther could even do more aggregations on the main query if required.</span>

In [13]:
-- The Top 10 was not required for the example but added to the main query to limit the file size before publishing to GitHub

USE Northwind2025;
GO

WITH MonthlyShipping (OrId, Id, [Month], ShippingCharge) AS (
    SELECT ord.OrderID
          ,ord.CustomerID
          ,month(ord.ShippedDate)
          ,sum(ord.Freight)
    FROM sales.Orders ord
    WHERE year(ord.ShippedDate) = 2024
    GROUP BY ord.OrderID
          ,ord.CustomerID
          ,month(ord.ShippedDate)

) 
, AvgShipping (Charge) AS (
    SELECT avg(ShippingCharge)  FROM MonthlyShipping
) 
, CustomerLUP (Id, Customer) AS (
    SELECT CustomerID
          ,CompanyName
    FROM sales.Customers
)
, MonthLup (iMth, tMth) AS (
    SELECT DISTINCT CalMonth, CalMonthName  FROM rpt.Calendar
)
SELECT TOP 10 with TIES ms.Id
      ,cus.Customer
      ,ms.OrId
      ,ms.ShippingCharge
FROM MonthlyShipping ms
    INNER JOIN CustomerLUP cus
     ON ms.Id = cus.Id
    INNER JOIN MonthLup mlup
     ON ms.[Month] = mlup.iMth
WHERE ms.ShippingCharge >= (Select * FROM AvgShipping)
ORDER BY ms.ShippingCharge desc
;



Commands completed successfully.

(10 rows affected)

Total execution time: 00:00:00.082

Id,Customer,OrId,ShippingCharge
QUICK,QUICK-Stop,10540,1007.64
QUICK,QUICK-Stop,10691,810.05
ERNSH,Ernst Handel,10514,789.95
RATTC,Rattlesnake Canyon Grocery,10479,708.95
SAVEA,Save-a-lot Markets,10612,544.08
FOLIG,Folies gourmandes,10634,487.38
ERNSH,Ernst Handel,10633,477.90
ERNSH,Ernst Handel,10430,458.78
QUICK,QUICK-Stop,10694,398.36
SAVEA,Save-a-lot Markets,10678,388.98
